# Module 3: Machine Learning Dataset Preparation

## Objective
Prepare an ML-ready dataset for EcoPackAI by combining product requirements and material properties.
This dataset will be used to train two regression models in Module 4:
- Cost Prediction
- CO₂ Impact Prediction

## Steps in this notebook
1. Load datasets (products and materials)
2. Perform basic data validation (nulls and duplicates)
3. Create ML dataset (product × material combinations)
4. Create target variables (cost and CO₂)
5. Select ML features (inputs)
6. Split into training and testing sets
7. Scale features (StandardScaler)
8. Save the prepared dataset for Module 4


In [1]:
import pandas as pd
import numpy as np


In [2]:
materials_path = "../data/processed/materials_dataset.csv"
products_path  = "../data/processed/products_dataset.csv"

materials_df = pd.read_csv(materials_path)
products_df  = pd.read_csv(products_path)

print("Materials shape:", materials_df.shape)
print("Products shape:", products_df.shape)

materials_df.head()


Materials shape: (120, 10)
Products shape: (175, 9)


,material_id,material_name,strength_score,weight_capacity_kg,biodegradability_score,co2_emission_kg,recyclability_percent,cost_per_unit_inr,product_category,used_for_products
0,M001,Single-wall corrugated cardboard,8,15,9,1.6,92,55,Electronics,Shipping boxes
1,M002,Double-wall corrugated cardboard,9,20,8,2.0,90,75,Electronics,Heavy-duty boxes
2,M003,Triple-wall corrugated cardboard,10,30,7,2.5,88,95,Industrial,Machinery packaging
3,M004,Kraft linerboard,7,12,9,1.5,90,50,Food,Outer cartons
4,M005,Test linerboard,7,10,8,1.8,85,48,Retail,Packaging cartons


In [3]:
print("Materials columns:")
print(list(materials_df.columns))

print("\nProducts columns:")
print(list(products_df.columns))


Materials columns:
['material_id', 'material_name', 'strength_score', 'weight_capacity_kg', 'biodegradability_score', 'co2_emission_kg', 'recyclability_percent', 'cost_per_unit_inr', 'product_category', 'used_for_products']

Products columns:
['product_id', 'product_name', 'product_category', 'product_weight_kg', 'fragility_level', 'required_strength_score', 'preferred_biodegradability_score', 'max_packaging_cost_inr', 'temperature_sensitive']


In [4]:
print("Missing values in materials:")
display(materials_df.isnull().sum())

print("\nMissing values in products:")
display(products_df.isnull().sum())

print("\nDuplicate rows in materials:", materials_df.duplicated().sum())
print("Duplicate rows in products:", products_df.duplicated().sum())


Missing values in materials:


material_id               0
material_name             0
strength_score            0
weight_capacity_kg        0
biodegradability_score    0
co2_emission_kg           0
recyclability_percent     0
cost_per_unit_inr         0
product_category          0
used_for_products         0
dtype: int64


Missing values in products:


product_id                          0
product_name                        0
product_category                    0
product_weight_kg                   0
fragility_level                     0
required_strength_score             0
preferred_biodegradability_score    0
max_packaging_cost_inr              0
temperature_sensitive               0
dtype: int64


Duplicate rows in materials: 0
Duplicate rows in products: 0


In [5]:
materials_df = materials_df.drop_duplicates()
products_df  = products_df.drop_duplicates()

print("After removing duplicates:")
print("Materials:", materials_df.shape)
print("Products:", products_df.shape)


After removing duplicates:
Materials: (120, 10)
Products: (175, 9)


In [6]:
# Create cross join between products and materials
materials_df["_key"] = 1
products_df["_key"] = 1

ml_df = products_df.merge(materials_df, on="_key").drop(columns=["_key"])

print("ML dataset shape:", ml_df.shape)
ml_df.head()


ML dataset shape: (21000, 19)


,product_id,product_name,product_category_x,product_weight_kg,fragility_level,required_strength_score,preferred_biodegradability_score,max_packaging_cost_inr,temperature_sensitive,material_id,material_name,strength_score,weight_capacity_kg,biodegradability_score,co2_emission_kg,recyclability_percent,cost_per_unit_inr,product_category_y,used_for_products
0,P001,Smartphone,Electronics,0.22,High,7,7,100,No,M001,Single-wall corrugated cardboard,8,15,9,1.6,92,55,Electronics,Shipping boxes
1,P001,Smartphone,Electronics,0.22,High,7,7,100,No,M002,Double-wall corrugated cardboard,9,20,8,2.0,90,75,Electronics,Heavy-duty boxes
2,P001,Smartphone,Electronics,0.22,High,7,7,100,No,M003,Triple-wall corrugated cardboard,10,30,7,2.5,88,95,Industrial,Machinery packaging
3,P001,Smartphone,Electronics,0.22,High,7,7,100,No,M004,Kraft linerboard,7,12,9,1.5,90,50,Food,Outer cartons
4,P001,Smartphone,Electronics,0.22,High,7,7,100,No,M005,Test linerboard,7,10,8,1.8,85,48,Retail,Packaging cartons


In [7]:
ml_df["target_cost_inr"] = (
    ml_df["cost_per_unit_inr"] +
    (ml_df["product_weight_kg"] * 10)
)

ml_df[["material_name", "product_name", "target_cost_inr"]].head()


,material_name,product_name,target_cost_inr
0,Single-wall corrugated cardboard,Smartphone,57.2
1,Double-wall corrugated cardboard,Smartphone,77.2
2,Triple-wall corrugated cardboard,Smartphone,97.2
3,Kraft linerboard,Smartphone,52.2
4,Test linerboard,Smartphone,50.2


In [8]:
ml_df["target_co2_kg"] = (
    ml_df["co2_emission_kg"] +
    (ml_df["product_weight_kg"] * 0.5)
)

ml_df[["material_name", "product_name", "target_co2_kg"]].head()


,material_name,product_name,target_co2_kg
0,Single-wall corrugated cardboard,Smartphone,1.71
1,Double-wall corrugated cardboard,Smartphone,2.11
2,Triple-wall corrugated cardboard,Smartphone,2.61
3,Kraft linerboard,Smartphone,1.61
4,Test linerboard,Smartphone,1.91


In [9]:
ml_df[["target_cost_inr", "target_co2_kg"]].describe()


,target_cost_inr,target_co2_kg
count,21000.000000,21000.000000
mean,129.866714,4.356252
std,61.036961,2.457207
min,38.600000,0.930000
25%,90.000000,2.450000
50%,112.500000,3.875000
75%,156.000000,5.575000
max,550.000000,20.000000


In [10]:
ml_features = [
    "product_weight_kg",
    "required_strength_score",
    "preferred_biodegradability_score",
    "strength_score",
    "weight_capacity_kg",
    "biodegradability_score",
    "recyclability_percent",
    "co2_emission_kg",
    "cost_per_unit_inr"
]

X = ml_df[ml_features].copy()
y_cost = ml_df["target_cost_inr"].copy()
y_co2  = ml_df["target_co2_kg"].copy()

print("X shape:", X.shape)
print("y_cost shape:", y_cost.shape)
print("y_co2 shape:", y_co2.shape)


X shape: (21000, 9)
y_cost shape: (21000,)
y_co2 shape: (21000,)


In [11]:
from sklearn.model_selection import train_test_split

X_train_cost, X_test_cost, y_train_cost, y_test_cost = train_test_split(
    X, y_cost, test_size=0.2, random_state=42
)

X_train_co2, X_test_co2, y_train_co2, y_test_co2 = train_test_split(
    X, y_co2, test_size=0.2, random_state=42
)

print("Cost Train:", X_train_cost.shape, "Cost Test:", X_test_cost.shape)
print("CO2 Train:", X_train_co2.shape, "CO2 Test:", X_test_co2.shape)


Cost Train: (16800, 9) Cost Test: (4200, 9)
CO2 Train: (16800, 9) CO2 Test: (4200, 9)


In [12]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Scale cost dataset
X_train_cost_scaled = scaler.fit_transform(X_train_cost)
X_test_cost_scaled  = scaler.transform(X_test_cost)

# Scale CO2 dataset
X_train_co2_scaled = scaler.fit_transform(X_train_co2)
X_test_co2_scaled  = scaler.transform(X_test_co2)

print("Feature scaling completed successfully ✅")


Feature scaling completed successfully ✅


In [13]:
output_path = "../data/processed/ml_dataset_module3.csv"
ml_df.to_csv(output_path, index=False)

print("ML dataset saved at:", output_path)


ML dataset saved at: ../data/processed/ml_dataset_module3.csv


## Module 3 Completion Summary ✅

- Loaded and validated products and materials datasets
- Performed data quality checks (nulls and duplicates)
- Created an ML dataset with ~21,000 product–material combinations
- Engineered target variables:
  - `target_cost_inr` for cost prediction
  - `target_co2_kg` for CO₂ impact prediction
- Selected relevant numerical features for ML
- Split data into training and testing sets (80/20)
- Applied feature scaling using StandardScaler
- Saved ML-ready dataset for use in Module 4

Module 3 successfully prepares the data pipeline for supervised machine learning.
